# 26 Pengaruh Initializer terhadap Profil Semantik


In [1]:
from pathlib import Path
import os, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
ROOT_26 = Path(os.environ.get('SEMANTIC26_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_26 / 'LIB').is_dir():
    raise FileNotFoundError('Folder LIB proyek belum tersedia: ' + str(ROOT_26))
print('Proyek:', ROOT_26)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Protokol eksperimen

| Analisis | Masukan | Hasil |
|---|---|---|
| Profil setiap initializer | 64 partisi LMS full, 240 LMS resample, 64 LMS train, 64 Pendikom train | Mapping semantik, evidence mode/anggota, dan bank target train/test |
| Kesesuaian ILS | 322 train / 139 test Pendikom | Accuracy, balanced accuracy, macro-F1, confusion matrix, counts, coverage, dan CI |
| Stabilitas LMS | 30 resample 80% yang sudah ditetapkan | Agreement arah, Base LS, unresolved, keragaman profil, ARI/NMI/AMI |
| Pelaporan | Hasil per run dan per peserta | Ringkasan delapan metode, tabel, gambar, dan artefak untuk tahap berikutnya |

Aturan utama tetap mode-first, member evidence sebagai pemecah seri, dan
magnitudo dari distribusi evidence anggota pada skala behavioral 11 unit.
Evidence LMS mencakup 13 fitur; Pendikom memakai 18 fitur retained.
Mapping holdout dibentuk hanya dari train; test tidak mengubah mapping.
Label ILS digunakan untuk evaluasi, bukan pemilihan seed atau aturan.

Metode stokastik memakai seluruh 15 run; metode deterministik satu run.
Referensi stabilitas adalah profil full run 0 milik masing-masing
initializer sesuai deklarasi tahap 25. Nomor cluster bukan label semantik.


In [2]:
KONFIGURASI_26 = {
    'bootstrap_repetitions': 10000,
    'bootstrap_seed': 20261014,
    'save_member_comparisons': True,
    'force': False,
}


## Ketidakpastian dan kasus unresolved

Ukuran utama adalah mean balanced accuracy Pendikom pada empat dimensi.
Bootstrap 95% mengambil ulang **peserta yang sama untuk semua metode dan
run**, dengan empat dimensi peserta tetap bersama. Metrik dihitung per run
lalu dirata-ratakan; variasi seed dilaporkan terpisah. CI bersyarat pada
train/split dan kumpulan mode yang dibekukan.

Prediksi unresolved tetap masuk denominator 139 peserta sebagai abstention,
dengan coverage terpisah. Macro-F1 dihitung untuk dua kutub reference;
unresolved menambah kesalahan recall. Unresolved berbeda dari kategori
magnitudo Balanced 1 atau 3. Pada stabilitas, unresolved tidak dihitung
sebagai arah yang cocok; agreement resolved-only disertai coverage.

Selisih CARE-KM terhadap setiap baseline memakai bootstrap peserta yang
berpasangan. CI individual bersifat deskriptif. Tujuh kontras ukuran utama
mean BA juga diberi interval percentile Bonferroni untuk keluarga tersebut.
Tiga puluh resample LMS saling tumpang tindih dan diringkas secara
deskriptif, tanpa p-value yang memperlakukannya sebagai cohort independen.

Jalankan ulang notebook ini jika Colab terputus. Cache per partisi,
stabilitas, evaluasi ILS, dan gambar memeriksa isi, kode, parameter, dan
versi lingkungan. Perubahan tampilan output notebook tidak membatalkan
cache. `force=True` menghitung ulang interpretasi/evaluasi dari bank 25.


In [3]:
SEMANTIC26_SOURCE = '"""Fixed-rule semantic evaluation from the immutable partition bank of stage 25."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport ast\nimport gzip\nimport hashlib\nimport importlib.metadata\nimport io\nimport itertools\nimport json\nimport os\nimport platform\nimport re\nimport sys\nimport uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE = \'26_Pengaruh_Initializer_Profil_Semantik\'\nUPSTREAM = \'25_Perbandingan_Initializer_Terintegrasi\'\nUP = \'OUTPUT/TAMBAHAN/\' + UPSTREAM\nUP_CONTROL = \'OUTPUT/TAMBAHAN/KONTROL/\' + UPSTREAM\nMETHODS = [\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\nSTOCHASTIC = {\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\nORDER = [\'CARE-KM\',\'Cao\',\'Bai\',\'BK-Modes\',\'K-Modes++\',\'Random\',\'Huang\',\'HR-2023\']\nKEYS = [\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\nDIMS = [\'Active/Reflective\',\'Sensing/Intuitive\',\'Visual/Verbal\',\'Sequential/Global\']\nPOLES = [(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\nABBR = {\'Active\':\'Act.\',\'Reflective\':\'Ref.\',\'Sensing\':\'Sens.\',\'Intuitive\':\'Int.\',\n        \'Visual\':\'Vis.\',\'Verbal\':\'Ver.\',\'Sequential\':\'Seq.\',\'Global\':\'Gl.\'}\nLS_LOOKUP = {combo:\'LS%02d\' % (i+1) for i,combo in enumerate(itertools.product(*POLES))}\nMAIN = [\'content_visit\',\'content_stay\',\'selfass_visit_proxy\',\'selfass_stay_proxy\',\'exercise_visit_proxy\',\n        \'exercise_stay_proxy\',\'forum_visit\',\'forum_stay\',\'forum_post\',\'quiz_stay_results_proxy\',\n        \'navigation_overview_visit\',\'navigation_overview_stay\',\'return_to_overview_rate\']\nPEND = [\'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\'exercise_visit\',\'exercise_stay\',\n        \'ques_detail\',\'ques_overview\',\'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\'ques_interpret\',\n        \'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\'navigation_overview_visit\',\'navigation_overview_stay\']\nFULL = \'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv\'\nRULE_FILES = {\'LMS\':\'OUTPUT/EKSPERIMEN/SEMANTIK/11_Interpretasi_Semantik_FSLSM/08_semantic_evidence_matrix.csv\',\n              \'PENDIKOM\':\'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_semantic_rules.csv\'}\nDEFAULT_CONFIG = {\'bootstrap_repetitions\':10000,\'bootstrap_seed\':20261014,\n                  \'save_member_comparisons\':True,\'force\':False}\nMETRIC_NAMES = [\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(value):\n    return sha(json.dumps(clean(value),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode(\'utf-8\'))\n\n\ndef clean(value):\n    if isinstance(value,dict): return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple,np.ndarray)): return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)): return int(value)\n    if isinstance(value,(float,np.floating)): return float(value) if np.isfinite(value) else None\n    if isinstance(value,np.bool_): return bool(value)\n    return value\n\n\ndef atomic(path, data):\n    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    temp = path.with_name(path.name+\'.\'+uuid.uuid4().hex[:8]+\'.tmp\')\n    temp.write_bytes(data); temp.replace(path)\n\n\ndef save_json(path, value):\n    atomic(path,json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False).encode(\'utf-8\'))\n\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False,lineterminator=\'\\n\',float_format=\'%.17g\').encode(\'utf-8\')\n    atomic(path,gzip.compress(data,mtime=0) if str(path).endswith(\'.gz\') else data)\n\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        return obj if isinstance(obj,dict) else {}\n    except (OSError,ValueError,UnicodeError): return {}\n\n\nclass Files:\n    def __init__(self, root):\n        self.root = Path(root).resolve(); self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root): raise ValueError(\'Lokasi di luar proyek: \'+relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative); before = path.stat()\n        stamp = (before.st_size,before.st_mtime_ns,before.st_ctime_ns)\n        if relative in self.cache and self.cache[relative][0] == stamp: return self.cache[relative][1]\n        data = path.read_bytes(); after = path.stat()\n        if stamp != (after.st_size,after.st_mtime_ns,after.st_ctime_ns):\n            raise RuntimeError(\'Berkas berubah ketika dibaca: \'+relative)\n        self.cache[relative] = (stamp,data); return data\n\n    def hash(self, relative): return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith(\'.gz\'): data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data),dtype=str if strings else None,\n                           keep_default_na=not strings,float_precision=\'round_trip\')\n\n\ndef runtime():\n    packages = {}\n    for name in [\'numpy\',\'pandas\',\'scipy\',\'sklearn\',\'matplotlib\']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod,\'__version__\',None) or importlib.metadata.version(\'scikit-learn\' if name==\'sklearn\' else name)\n    return {\'python\':platform.python_version(),\'packages\':packages}\n\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read(\'PYTHON/\'+stem+\'.ipynb\').decode(\'utf-8-sig\')); code=[]\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']!=\'code\': continue\n        source=\'\'.join(cell[\'source\'])\n        if tag in cell.get(\'metadata\',{}).get(\'tags\',[]):\n            tree=ast.parse(source)\n            if (len(tree.body)!=1 or not isinstance(tree.body[0],ast.Assign)\n                or len(tree.body[0].targets)!=1 or not isinstance(tree.body[0].targets[0],ast.Name)\n                or tree.body[0].targets[0].id!=variable or not isinstance(ast.literal_eval(tree.body[0].value),dict)):\n                raise RuntimeError(\'Sel konfigurasi hanya boleh berisi literal \'+variable)\n        else: code.append(source)\n    return digest(code)\n\n\ndef expected_rules(dataset):\n    rows=[]\n    def add(features,dim,low,high):\n        for f in features.split(): rows.append((f,dim,low,high))\n    if dataset==\'LMS\':\n        add(\'content_visit content_stay selfass_stay_proxy forum_visit quiz_stay_results_proxy\',DIMS[0],*POLES[0])\n        add(\'selfass_visit_proxy exercise_visit_proxy exercise_stay_proxy forum_post\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay\',DIMS[1],*POLES[1])\n        add(\'selfass_visit_proxy selfass_stay_proxy exercise_visit_proxy quiz_stay_results_proxy\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit forum_visit forum_stay forum_post\',DIMS[2],*POLES[2])\n        add(\'navigation_overview_visit navigation_overview_stay return_to_overview_rate\',DIMS[3],*POLES[3])\n    else:\n        add(\'selfass_visit selfass_twice_wrong exercise_visit exercise_stay\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay quiz_stay_results\',DIMS[0],*POLES[0])\n        add(\'selfass_visit exercise_visit ques_detail ques_facts quiz_revisions quiz_stay_results\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit content_stay ques_concepts ques_develop\',DIMS[1],*POLES[1])\n        add(\'ques_graphics\',DIMS[2],*POLES[2][::-1]); add(\'content_visit ques_text\',DIMS[2],*POLES[2])\n        add(\'ques_detail\',DIMS[3],*POLES[3][::-1])\n        add(\'ques_overview ques_interpret ques_develop navigation_overview_visit navigation_overview_stay\',DIMS[3],*POLES[3])\n    return sorted(rows)\n\n\ndef check_config(configuration):\n    if set(configuration)-set(DEFAULT_CONFIG): raise ValueError(\'Konfigurasi tidak dikenal.\')\n    cfg=dict(DEFAULT_CONFIG);cfg.update(configuration)\n    for key in [\'bootstrap_repetitions\',\'bootstrap_seed\']:\n        if isinstance(cfg[key],bool) or not isinstance(cfg[key],int) or cfg[key]<0: raise ValueError(key+\' tidak valid.\')\n    if cfg[\'bootstrap_repetitions\']<100: raise ValueError(\'Gunakan sedikitnya 100 bootstrap untuk diagnostik; default publikasi 10000.\')\n    for key in [\'save_member_comparisons\',\'force\']:\n        if not isinstance(cfg[key],bool): raise ValueError(key+\' harus True/False.\')\n    return cfg\n\n\ndef input_setup(files):\n    required = [UP+\'/PROTOKOL/02_konfigurasi.json\',UP_CONTROL+\'/status_eksekusi.json\',\n                FULL,*RULE_FILES.values(),\'LIB/pipeline_data.py\',\'LIB/pendikom_ils_labels.csv\',\n                \'OUTPUT/SPLIT/LMS/train_categorizer_state.json\',\n                \'OUTPUT/SPLIT/LMS/70_30_main_lms_train_only_thresholds.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',\n                \'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\',\n                \'PYTHON/\'+UPSTREAM+\'.ipynb\',\'PYTHON/\'+STAGE+\'.ipynb\']\n    for ds in [\'LMS\',\'PENDIKOM\']:\n        required += [f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n    required += [f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv\' for r in range(30)]\n    missing=[p for p in required if not files.path(p).is_file()]\n    if missing: raise FileNotFoundError(\'Masukan belum tersedia. Selesaikan tahap 25 dan pertahankan hasil 01–24:\\n\'+\'\\n\'.join(missing[:20]))\n    status=json.loads(files.read(UP_CONTROL+\'/status_eksekusi.json\'))\n    if status.get(\'status\')!=\'selesai\' or status.get(\'full_protocol\') is not True or status.get(\'n_valid\')!=3272:\n        raise RuntimeError(\'Tahap 25 belum selesai dengan protokol lengkap dan seluruh run valid.\')\n    protocol=json.loads(files.read(UP+\'/PROTOKOL/02_konfigurasi.json\'))\n    if protocol[\'constants\'] != {\'FROZEN_CARE_CONFIG\':{\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0},\'K\':16,\'MAX_ITER\':100,\'SI_SAMPLE_SIZE\':2000}:\n        raise RuntimeError(\'Konfigurasi tahap 25 berbeda dari protokol yang ditetapkan.\')\n    source_code=protocol[\'code\']\n    for relative,checksum in source_code[\'project_libraries\'].items():\n        if files.hash(relative)!=checksum: raise RuntimeError(\'Pustaka sumber tahap 25 berubah: \'+relative)\n        required.append(relative)\n    if notebook_hash(files,UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\') != source_code[\'notebook_code\']:\n        raise RuntimeError(\'Kode notebook 25 berubah setelah eksperimen. Gunakan notebook sumber yang sesuai.\')\n    literals={}\n    nb=json.loads(files.read(\'PYTHON/\'+UPSTREAM+\'.ipynb\'))\n    for cell in nb[\'cells\']:\n        if cell[\'cell_type\']==\'code\':\n            for node in ast.parse(\'\'.join(cell[\'source\'])).body:\n                if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and isinstance(node.value,ast.Constant):\n                    literals[node.targets[0].id]=node.value.value\n    for name,var in [(\'initializer\',\'BK_MODES_SOURCE\'),(\'verification\',\'VERIFICATION_SOURCE\'),(\'comparison\',\'COMPARISON_SOURCE\')]:\n        if sha(literals[var].encode(\'utf-8\'))!=source_code[\'embedded_sources\'][name]: raise RuntimeError(\'Sumber tertanam tahap 25 tidak sesuai.\')\n    feature_state={}\n    for node in ast.parse(files.read(\'LIB/pipeline_data.py\').decode(\'utf-8-sig\')).body:\n        if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name) and node.targets[0].id in [\'MAIN_FEATURES\',\'PENDIKOM_FEATURES\']:\n            feature_state[node.targets[0].id]=ast.literal_eval(node.value)\n    if feature_state[\'MAIN_FEATURES\']!=MAIN or [x for x in feature_state[\'PENDIKOM_FEATURES\'] if x!=\'navigation_skip\']!=PEND:\n        raise RuntimeError(\'Urutan fitur tidak sesuai protokol.\')\n    audit=files.frame(\'OUTPUT/SPLIT/PENDIKOM/70_30_groundtruth_feature_audit.csv\',True)\n    if audit.loc[audit.used.str.lower().eq(\'true\'),\'feature\'].tolist()!=PEND: raise RuntimeError(\'Mask fitur train Pendikom berubah.\')\n    rules={}\n    for ds,path in RULE_FILES.items():\n        frame=files.frame(path,True)\n        cols=[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']\n        if sorted(map(tuple,frame[cols].to_numpy())) != expected_rules(ds): raise RuntimeError(\'Evidence rules berbeda dari aturan utama: \'+ds)\n        if \'moderate_supports\' in frame and not frame.moderate_supports.eq(\'Neutral\').all(): raise RuntimeError(\'Evidence moderate harus netral.\')\n        rules[ds]=frame[cols]\n    watched={p:files.hash(p) for p in set(required) if not p.endswith(\'.ipynb\') and not p.endswith(\'status_eksekusi.json\')}\n    return protocol,rules,watched\n\n\ndef contexts(files):\n    def make(scope,ds,frame,features,idcol,source,rep=None,idx=None,test=None):\n        if frame[idcol].duplicated().any(): raise RuntimeError(\'Identitas peserta tidak unik: \'+scope)\n        X=frame[features].to_numpy(dtype=object)\n        if not np.isin(X,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori di luar 1/2/3: \'+scope)\n        indices=np.arange(len(X),dtype=np.int64) if idx is None else idx\n        ids=frame[idcol].tolist();testids=test[idcol].tolist() if test is not None else []\n        if len(set(testids))!=len(testids) or set(ids)&set(testids): raise RuntimeError(\'Train/test tumpang tindih atau ID test berulang.\')\n        Xt=test[features].to_numpy(dtype=object) if test is not None else None\n        matrix=digest({\'features\':features,\'rows\':X.tolist(),\'ids\':ids,\'indices\':indices.tolist(),\n                       \'test_rows\':Xt.tolist() if Xt is not None else None,\'test_ids\':testids,\'classes\':None})\n        return {\'scope\':scope,\'dataset\':ds,\'X\':X,\'ids\':ids,\'indices\':indices,\'test_X\':Xt,\'test_ids\':testids,\n                \'features\':features,\'sources\':source,\'rep\':rep,\'matrix_sha256\':matrix,\'test_frame\':test}\n    frame=files.frame(FULL,True)\n    if frame.shape!=(6658,14) or frame.columns.tolist()!=[\'user\']+MAIN: raise RuntimeError(\'Matriks full LMS berbeda.\')\n    yield make(\'LMS_FULL\',\'LMS\',frame,MAIN,\'user\',[FULL])\n    for rep in range(30):\n        path=f\'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv\'\n        idx=files.frame(path)[\'source_row_index\'].to_numpy(np.int64)\n        if len(idx)!=5326 or len(np.unique(idx))!=5326 or np.any(idx<0) or np.any(idx>=6658): raise RuntimeError(\'Indeks resample tidak valid.\')\n        yield make(\'LMS_RESAMPLING\',\'LMS\',frame.iloc[idx],MAIN,\'user\',[FULL,path],rep,idx)\n    for ds,ntr,nte,features,idcol in [(\'LMS\',4660,1998,MAIN,\'user\'),(\'PENDIKOM\',322,139,PEND,\'userid\')]:\n        paths=[f\'OUTPUT/SPLIT/{ds}/{part}/categorical.csv\' for part in [\'TRAIN\',\'TEST\']]\n        tr,te=[files.frame(p,True) for p in paths]\n        if len(tr)!=ntr or len(te)!=nte: raise RuntimeError(\'Ukuran train/test berbeda: \'+ds)\n        if \'split\' in tr and (not tr.split.str.lower().eq(\'train\').all() or not te.split.str.lower().eq(\'test\').all()): raise RuntimeError(\'Kolom split tidak sesuai.\')\n        yield make(ds+\'_TRAIN\',ds,tr,features,idcol,paths,test=te)\n\n\ndef job_identity(context,method,run):\n    return {\'scope\':context[\'scope\'],\'dataset\':context[\'dataset\'],\'rep\':context[\'rep\'],\'run\':run,\'method\':method}\n\n\ndef upstream_job(files,protocol,context,method,run):\n    identity=job_identity(context,method,run);key=digest(identity)[:24]\n    relative=UP_CONTROL+\'/JOBS/\'+key+\'.json\'\n    if not files.path(relative).is_file(): raise FileNotFoundError(\'Partisi tahap 25 belum tersedia: \'+str(identity))\n    record=json.loads(files.read(relative));checksum=record.pop(\'record_sha256\',\'\')\n    if digest(record)!=checksum or record.get(\'status\')!=\'selesai\' or record.get(\'identity\')!=identity:\n        raise RuntimeError(\'Catatan partisi tahap 25 tidak valid: \'+relative)\n    p=record[\'parameters\'];row=record[\'row\']\n    if record[\'signature\']!=digest({\'parameters\':p,\'code\':protocol[\'code\'],\'runtime\':protocol[\'runtime\']}): raise RuntimeError(\'Signature tahap 25 tidak sesuai protokol.\')\n    expected_seed=(6161600+context[\'rep\']) if context[\'rep\'] is not None else 4161600+run\n    if (p[\'matrix_sha256\']!=context[\'matrix_sha256\'] or p[\'k\']!=16 or p[\'seed\']!=expected_seed\n        or p[\'max_iter\']!=100 or p[\'care_config\']!=protocol[\'constants\'][\'FROZEN_CARE_CONFIG\']\n        or not p[\'needs_partition\'] or row.get(\'valid\') is not True or not row.get(\'partition_available\')):\n        raise RuntimeError(\'Isi matriks/parameter/partisi tahap 25 tidak sesuai: \'+str(identity))\n    watched={relative:files.hash(relative)}\n    for path,expected in record[\'outputs\'].items():\n        if not path.startswith(UP+\'/BANK/\'): raise RuntimeError(\'Lokasi bank tahap 25 tidak sesuai.\')\n        if files.hash(path)!=expected: raise RuntimeError(\'Artefak tahap 25 berubah. Pulihkan/jalankan ulang 25: \'+path)\n        watched[path]=expected\n    def find(filename):\n        found=[p for p in record[\'outputs\'] if p.endswith(\'/\'+filename)]\n        if len(found)!=1: raise RuntimeError(\'Artefak wajib tidak unik: \'+filename)\n        return found[0]\n    modes=files.frame(find(\'final_modes.csv\'),True)\n    if modes.columns.tolist()!=[\'cluster_id\']+context[\'features\'] or modes.cluster_id.astype(int).tolist()!=list(range(16)):\n        raise RuntimeError(\'Skema/urutan mode akhir tidak sesuai.\')\n    modes=modes[context[\'features\']].to_numpy(dtype=object)\n    if not np.isin(modes,[\'1\',\'2\',\'3\']).all(): raise RuntimeError(\'Kategori mode tidak valid.\')\n    assignment=files.frame(find(\'train_assignments.csv\'),True)\n    if assignment.identity.tolist()!=context[\'ids\'] or not np.array_equal(assignment.source_row_index.astype(np.int64),context[\'indices\']):\n        raise RuntimeError(\'Identitas/urutan anggota tidak sesuai matriks.\')\n    labels=assignment.cluster_id.to_numpy(int)\n    if not np.isin(labels,np.arange(16)).all() or (np.bincount(labels,minlength=16)==0).any(): raise RuntimeError(\'Keanggotaan cluster tidak valid.\')\n    cost=int(np.sum(context[\'X\']!=modes[labels]))\n    if cost!=int(row[\'final_cost\']): raise RuntimeError(\'Cost anggota/mode berbeda dari hasil tahap 25.\')\n    testlabels=None\n    if context[\'test_X\'] is not None:\n        test=files.frame(find(\'test_assignments.csv\'),True)\n        if test.identity.tolist()!=context[\'test_ids\']: raise RuntimeError(\'Urutan identitas test berbeda.\')\n        testlabels=test.cluster_id.to_numpy(int)\n        distance=np.zeros((len(testlabels),16),dtype=np.int64)\n        for j in range(len(context[\'features\'])): distance += context[\'test_X\'][:,j,None]!=modes[None,:,j]\n        if not np.array_equal(testlabels,np.argmin(distance,axis=1)) or not np.array_equal(test.distance.astype(int),distance.min(axis=1)):\n            raise RuntimeError(\'Assignment test berbeda dari mode train yang dibekukan.\')\n    return {\'record\':record,\'checksum\':checksum,\'watched\':watched,\'modes\':modes,\'labels\':labels,\'test_labels\':testlabels,\'identity\':identity}\n\n\ndef interpret(X,modes,labels,features,rules):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    sizes=np.bincount(labels,minlength=16);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(16):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n\n\ndef profiles(mapping,labels,ids,indices=None):\n    cols=KEYS+[k+\'_score\' for k in KEYS]+[\'ls_class\',\'mapping_status\']\n    frame=mapping.set_index(\'cluster_id\').loc[labels,cols].reset_index()\n    frame.insert(0,\'identity\',ids)\n    if indices is not None: frame.insert(1,\'source_row_index\',indices)\n    return frame\n\n\ndef numeric_profiles(frame):\n    return np.column_stack([np.where(frame[k].eq(ABBR[p[0]]),1,np.where(frame[k].eq(ABBR[p[1]]),-1,0)) for k,p in zip(KEYS,POLES)])\n\n\ndef cache_read(files,path,signature,force=False):\n    record=read_checkpoint(path)\n    checksum=record.pop(\'record_sha256\',\'\')\n    try: valid=digest(record)==checksum\n    except (TypeError,ValueError): valid=False\n    if force or not valid or record.get(\'signature\')!=signature or record.get(\'status\')!=\'selesai\': return None\n    if not all(files.path(p).is_file() and files.hash(p)==h for p,h in record.get(\'outputs\',{}).items()): return None\n    return record\n\n\ndef cache_write(files,path,signature,outputs,extra):\n    record={\'status\':\'selesai\',\'signature\':signature,\'outputs\':{str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in outputs},**extra}\n    record[\'record_sha256\']=digest(record);save_json(path,record);return record\n\n\ndef semantic_job(files,output,control,context,upstream,rules,code_state,cfg):\n    identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream_record_sha256\':upstream[\'checksum\'],\n                      \'matrix_sha256\':context[\'matrix_sha256\'],\'evidence\':rules.to_dict(\'records\'),\'rule\':\'mode-first-member-tie-11eq-v1\'})\n    checkpoint=control/\'PROFIL\'/f\'{key}.json\'\n    record=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    folder=output/\'BANK\'/identity[\'scope\']/identity[\'dataset\']/(\'r%02d\' % (identity[\'rep\'] if identity[\'rep\'] is not None else identity[\'run\']))/re.sub(r\'[^A-Za-z0-9]+\',\'_\',identity[\'method\'].replace(\'++\',\'_PP\')).strip(\'_\')\n    mapping_path=folder/\'01_cluster_profiles.csv\';evidence_path=folder/\'02_cluster_dimension_evidence.csv\'\n    if record:\n        mapping=files.frame(str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'));evidence=files.frame(str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\'))\n        disposition=\'CACHE\'\n    else:\n        mapping,evidence=interpret(context[\'X\'],upstream[\'modes\'],upstream[\'labels\'],context[\'features\'],rules)\n        save_csv(mapping_path,mapping);save_csv(evidence_path,evidence);paths=[mapping_path,evidence_path]\n        if context[\'test_X\'] is not None:\n            for filename,frame in [(\'03_train_targets.csv\',profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])),\n                                   (\'04_test_targets.csv\',profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\']))]:\n                path=folder/filename;save_csv(path,frame);paths.append(path)\n        record=cache_write(files,checkpoint,signature,paths,{\'identity\':identity,\'upstream_artifacts\':upstream[\'watched\']})\n        disposition=\'MAP\'\n    weighted=sum(mapping.cluster_size*mapping.mapping_status.ne(\'RESOLVED\'))/len(context[\'X\'])\n    summary={**identity,\'n_learners\':len(context[\'X\']),\'n_clusters\':16,\'n_resolved_profiles\':mapping.loc[mapping.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique(),\n             \'unresolved_learner_rate\':weighted,\'mode_member_conflicts\':int((~evidence.mode_member_consistency.astype(bool)).sum()),\n             \'strong_conflicts\':int(evidence.strong_mode_member_conflict.astype(bool).sum()),\'mapping_file\':str(mapping_path.relative_to(files.root)).replace(\'\\\\\',\'/\'),\n             \'evidence_file\':str(evidence_path.relative_to(files.root)).replace(\'\\\\\',\'/\')}\n    return mapping,evidence,record,summary,disposition\n\n\ndef count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a,b,c,d,e,f=np.moveaxis(np.asarray(counts,float),-1,0)\n    pos=a+b+c;neg=d+e+f;total=pos+neg\n    with np.errstate(divide=\'ignore\',invalid=\'ignore\'):\n        accuracy=(a+e)/total;ba=.5*(a/pos+e/neg)\n        denp=2*a+d+b+c;denn=2*e+b+d+f\n        fp=np.divide(2*a,denp,out=np.zeros_like(a),where=denp>0)\n        fn=np.divide(2*e,denn,out=np.zeros_like(e),where=denn>0)\n    return np.stack([accuracy,ba,.5*(fp+fn)],axis=-1)\n\n\ndef confusion_masks(truth,prediction):\n    # prediction: run x learner x dimension; output learner x run x dimension x six cells.\n    t=truth[None,:,:];p=prediction\n    conditions=[(t==1)&(p==1),(t==1)&(p==-1),(t==1)&(p==0),\n                (t==-1)&(p==1),(t==-1)&(p==-1),(t==-1)&(p==0)]\n    return np.stack(conditions,axis=-1).transpose(1,0,2,3).astype(float)\n\n\ndef ils_reference(files,context):\n    path=\'OUTPUT/SPLIT/PENDIKOM/TEST/reference_ils.csv\';ref=files.frame(path,True)\n    test=context[\'test_frame\']\n    if ref.userid.tolist()!=context[\'test_ids\'] or ref.courseid.tolist()!=test.courseid.tolist(): raise RuntimeError(\'Reference ILS tidak sejajar dengan peserta test.\')\n    original=files.frame(\'LIB/pendikom_ils_labels.csv\',True).set_index([\'userid\',\'courseid\'])\n    original=original.loc[list(zip(ref.userid,ref.courseid))].reset_index()\n    cols=[\'ils_\'+k+\'_code\' for k in KEYS]+[\'ils_ls_class\']\n    if not original[cols].reset_index(drop=True).equals(ref[cols].reset_index(drop=True)) or not original.split.str.lower().eq(\'test\').all():\n        raise RuntimeError(\'Reference ILS test berbeda dari label independen yang dibekukan.\')\n    truth=[]\n    for key,poles in zip(KEYS,POLES):\n        column=ref[\'ils_\'+key+\'_code\'];first=column.str.startswith(ABBR[poles[0]]);second=column.str.startswith(ABBR[poles[1]])\n        if not (first^second).all(): raise RuntimeError(\'Label arah ILS tidak dikenal.\')\n        values=np.where(first,1,-1)\n        if len(np.unique(values))!=2: raise RuntimeError(\'Dua kutub reference harus terwakili untuk metrik BA.\')\n        truth.append(values)\n    return ref,np.column_stack(truth)\n\n\ndef evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg):\n    folder=output/\'ILS\';checkpoint=control/\'evaluasi_ils.json\'\n    signature=digest({\'code\':code_state,\'truth\':truth.tolist(),\'ids\':reference.userid.tolist(),\n        \'predictions\':{m:sha(np.asarray(predictions[m],dtype=np.int8).tobytes()) for m in METHODS},\n        \'bootstrap_repetitions\':cfg[\'bootstrap_repetitions\'],\'bootstrap_seed\':cfg[\'bootstrap_seed\'],\n        \'unit\':\'learner; all four dimensions and all runs/methods remain paired\'})\n    saved=cache_read(files,checkpoint,signature,cfg[\'force\'])\n    if saved:\n        print(\'Evaluasi ILS/CI: gunakan hasil yang isinya terverifikasi sama.\',flush=True)\n        return files.frame(str((folder/\'04_ringkasan_overall.csv\').relative_to(files.root)).replace(\'\\\\\',\'/\')),True\n    from threadpoolctl import threadpool_limits\n    n=len(truth);nr=cfg[\'bootstrap_repetitions\'];rng=np.random.default_rng(cfg[\'bootstrap_seed\'])\n    masks={m:confusion_masks(truth,np.asarray(predictions[m],dtype=np.int8)) for m in METHODS}\n    counts={m:masks[m].sum(axis=0) for m in METHODS}\n    points={m:count_metrics(counts[m]) for m in METHODS}\n    boot=np.empty((nr,len(METHODS),5,3),dtype=np.float64)\n    valid=np.zeros(nr,dtype=bool)\n    print(\'CI ILS:\',nr,\'bootstrap berpasangan pada\',n,\'peserta.\',flush=True)\n    with threadpool_limits(limits=1):\n        for start in range(0,nr,250):\n            stop=min(start+250,nr);weights=rng.multinomial(n,np.full(n,1/n),size=stop-start).astype(float)\n            positive=weights@(truth==1);negative=n-positive\n            valid[start:stop]=((positive>0)&(negative>0)).all(axis=1)\n            for mi,method in enumerate(METHODS):\n                source=masks[method];shape=source.shape\n                values=(weights@source.reshape(n,-1)).reshape(stop-start,shape[1],4,6)\n                metrics=count_metrics(values).mean(axis=1)\n                boot[start:stop,mi,:4,:]=metrics;boot[start:stop,mi,4,:]=metrics.mean(axis=1)\n    if valid.sum()<100: raise RuntimeError(\'Bootstrap valid kurang dari 100; interval belum dapat dihitung.\')\n    boot=boot[valid];paths=[];raw_rows=[];cm_rows=[];ref_rows=[];summary=[];overall=[];pred_rows=[]\n    actual_counts=np.stack([(truth==1).sum(axis=0),(truth==-1).sum(axis=0)],axis=-1)\n    for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n        for ci,pole in enumerate(poles): ref_rows.append({\'dimension\':key,\'reference_pole\':pole,\'n\':int(actual_counts[di,ci]),\'n_test\':n})\n    for mi,method in enumerate(METHODS):\n        arr=np.asarray(predictions[method],dtype=np.int8);point=points[method];nrun=len(arr)\n        for run in range(nrun):\n            for li,identity in enumerate(reference.userid):\n                row={\'method\':method,\'run\':run,\'seed\':4161600+run,\'identity\':identity,\'courseid\':reference.courseid.iloc[li],\n                     \'reference_ls_class\':reference.ils_ls_class.iloc[li]}\n                full=[]\n                for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                    row[\'reference_\'+key]=poles[0] if truth[li,di]==1 else poles[1]\n                    value=arr[run,li,di];pole=poles[0] if value==1 else poles[1] if value==-1 else None\n                    row[\'predicted_\'+key]=pole or \'UNRESOLVED\';full.append(pole)\n                row[\'predicted_ls_class\']=LS_LOOKUP[tuple(full)] if all(full) else \'AMBIGUOUS\';pred_rows.append(row)\n            for di,(key,poles) in enumerate(zip(KEYS,POLES)):\n                cc=counts[method][run,di];a,b,c,d,e,f=cc;resolved=n-c-f\n                raw_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'n_test\':n,\'reference_first\':int(actual_counts[di,0]),\n                                 \'reference_second\':int(actual_counts[di,1]),\'n_resolved\':int(resolved),\'n_unresolved\':int(c+f),\n                                 \'coverage\':resolved/n,\'resolved_only_accuracy\':(a+e)/resolved if resolved else np.nan,\n                                 **dict(zip(METRIC_NAMES,point[run,di]))})\n                for ci,num in enumerate(cc):\n                    cm_rows.append({\'method\':method,\'run\':run,\'dimension\':key,\'reference_pole\':poles[ci//3],\n                                    \'predicted_pole\':(poles+(\'UNRESOLVED\',))[ci%3],\'n\':int(num)})\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            values=point[:,di,:] if di<4 else point.mean(axis=1)\n            row={\'method\':method,\'dimension\':key,\'n_runs\':nrun,\'n_test\':n,\n                 \'bootstrap_valid\':int(valid.sum()),\'bootstrap_requested\':nr,\n                 \'coverage_mean\':float((arr[:,:,di]!=0).mean()) if di<4 else float((arr!=0).mean()),\n                 \'complete_profile_coverage\':float((arr!=0).all(axis=2).mean())}\n            for ji,metric in enumerate(METRIC_NAMES):\n                low,high=np.quantile(boot[:,mi,di,ji],[.025,.975])\n                row[metric+\'_mean\']=float(values[:,ji].mean());row[metric+\'_sd_runs\']=float(values[:,ji].std(ddof=1)) if nrun>1 else np.nan\n                row[metric+\'_ci95_low\']=low;row[metric+\'_ci95_high\']=high\n            (summary if di<4 else overall).append(row)\n    differences=[];care=METHODS.index(\'CARE-KM\')\n    for mi,method in enumerate(METHODS):\n        if method==\'CARE-KM\': continue\n        for di,key in enumerate(KEYS+[\'mean_four_dimensions\']):\n            for ji,metric in enumerate(METRIC_NAMES):\n                delta=boot[:,care,di,ji]-boot[:,mi,di,ji];lo,hi=np.quantile(delta,[.025,.975])\n                carepoint=points[\'CARE-KM\'][:,di,ji].mean() if di<4 else points[\'CARE-KM\'][:,:,ji].mean()\n                basepoint=points[method][:,di,ji].mean() if di<4 else points[method][:,:,ji].mean()\n                row={\'baseline\':method,\'dimension\':key,\'metric\':metric,\'care_minus_baseline\':carepoint-basepoint,\n                     \'paired_ci95_low\':lo,\'paired_ci95_high\':hi,\'positive_favors\':\'CARE-KM\',\'n_learners\':n,\n                     \'interval\':\'paired learner percentile bootstrap; individual 95% intervals\'}\n                if di==4 and metric==\'balanced_accuracy\':\n                    low,high=np.quantile(delta,[.05/(2*7),1-.05/(2*7)])\n                    row.update(primary_family=\'seven overall mean-BA contrasts\',bonferroni_family95_low=low,bonferroni_family95_high=high)\n                differences.append(row)\n    outputs=[(\'01_prediksi_per_peserta.csv\',pd.DataFrame(pred_rows)),(\'02_metrik_per_dimensi_per_run.csv\',pd.DataFrame(raw_rows)),\n             (\'03_ringkasan_per_dimensi.csv\',pd.DataFrame(summary)),(\'04_ringkasan_overall.csv\',pd.DataFrame(overall)),\n             (\'05_confusion_matrices.csv\',pd.DataFrame(cm_rows)),(\'07_perbedaan_berpasangan_vs_care.csv\',pd.DataFrame(differences)),\n             (\'08_reference_counts.csv\',pd.DataFrame(ref_rows))]\n    for filename,frame in outputs:\n        path=folder/filename;save_csv(path,frame);paths.append(path)\n    buffer=io.BytesIO();np.savez_compressed(buffer,metrics=boot,methods=np.asarray(METHODS),dimensions=np.asarray(KEYS+[\'mean_four_dimensions\']),\n                                          metric_names=np.asarray(METRIC_NAMES),valid_draws=valid,seed=cfg[\'bootstrap_seed\'])\n    path=folder/\'06_bootstrap_paired_metrics.npz\';atomic(path,buffer.getvalue());paths.append(path)\n    path=folder/\'09_protokol_evaluasi.json\'\n    save_json(path,{\'primary_metric\':\'mean balanced accuracy across four dimensions\',\'n_learners\':n,\'n_bootstrap_valid\':int(valid.sum()),\n        \'seed\':cfg[\'bootstrap_seed\'],\'unit\':\'learner; all dimensions, runs, and methods paired\',\n        \'stochastic_summary\':\'metric per run, then arithmetic mean; no best-run selection\',\n        \'interval_scope\':\'conditional on frozen training split, modes, mapping, and saved initialization runs\',\n        \'unresolved\':\'included as incorrect/abstention; two reference poles used for macro-F1; coverage separately reported\',\n        \'multiplicity\':\'individual paired CIs descriptive; primary seven overall mean-BA contrasts also have Bonferroni percentile family intervals\',\n        \'semantic_scope\':\'direction correspondence; behavioral 11-unit magnitudes are not questionnaire scores\'})\n    paths.append(path)\n    cache_write(files,checkpoint,signature,paths,{\'n_test\':n,\'n_methods\':8,\'bootstrap_valid\':int(valid.sum())})\n    return pd.DataFrame(overall),False\n\n\ndef stability_job(files,output,control,context,upstream,mapping,reference,code_state,cfg):\n    from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score,adjusted_mutual_info_score\n    method=upstream[\'identity\'][\'method\'];rep=context[\'rep\'];identity=upstream[\'identity\'];key=digest(identity)[:24]\n    signature=digest({\'code\':code_state,\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\'),\n        \'reference\':reference[\'signature\'],\'save_members\':cfg[\'save_member_comparisons\']})\n    path=control/\'STABILITAS\'/f\'{key}.json\';cached=cache_read(files,path,signature,cfg[\'force\'])\n    if cached: return cached[\'summary\'],\'CACHE\'\n    idx=context[\'indices\'];current=profiles(mapping,upstream[\'labels\'],context[\'ids\'],idx)\n    ref=reference[\'profiles\'].iloc[idx].reset_index(drop=True)\n    if ref.identity.tolist()!=context[\'ids\']: raise RuntimeError(\'Peserta resample tidak sejajar dengan full reference initializer.\')\n    truth=numeric_profiles(ref);pred=numeric_profiles(current);paired=(truth!=0)&(pred!=0);agreement=paired&(truth==pred)\n    complete=agreement.all(axis=1)\n    row={**identity,\'n_learners\':len(idx),\'reference_full_run\':0,\'reference_seed\':4161600,\n        \'mean_direction_agreement\':float(agreement.mean()),\'all_four_direction_agreement\':float(complete.mean()),\'base_ls_agreement\':float(complete.mean()),\n        \'reference_complete_coverage\':float((truth!=0).all(axis=1).mean()),\'resample_complete_coverage\':float((pred!=0).all(axis=1).mean()),\n        \'unresolved_learner_rate\':float((pred==0).any(axis=1).mean()),\'joint_resolved_dimensional_coverage\':float(paired.mean()),\n        \'n_resolved_profiles\':int(current.loc[current.mapping_status.eq(\'RESOLVED\'),\'ls_class\'].nunique()),\n        \'ari\':float(adjusted_rand_score(ref.cluster_id,current.cluster_id)),\n        \'nmi\':float(normalized_mutual_info_score(ref.cluster_id,current.cluster_id)),\n        \'ami\':float(adjusted_mutual_info_score(ref.cluster_id,current.cluster_id))}\n    for di,keydim in enumerate(KEYS):\n        row[keydim+\'_agreement\']=float(agreement[:,di].mean())\n        row[keydim+\'_joint_resolved_coverage\']=float(paired[:,di].mean())\n        row[keydim+\'_resolved_only_agreement\']=float(agreement[:,di].sum()/paired[:,di].sum()) if paired[:,di].sum() else np.nan\n    refscore=ref[[k+\'_score\' for k in KEYS]].to_numpy(float);score=current[[k+\'_score\' for k in KEYS]].to_numpy(float)\n    row[\'mean_absolute_signed_score_change\']=float(np.nanmean(np.abs(score-refscore))) if paired.any() else np.nan\n    paths=[]\n    if cfg[\'save_member_comparisons\']:\n        frame=pd.DataFrame({\'identity\':context[\'ids\'],\'source_row_index\':idx,\'reference_cluster_id\':ref.cluster_id,\n                            \'resample_cluster_id\':current.cluster_id,\'reference_ls_class\':ref.ls_class,\'resample_ls_class\':current.ls_class,\n                            \'all_four_agreement\':complete})\n        for di,k in enumerate(KEYS):\n            frame[k+\'_reference\']=ref[k];frame[k+\'_resample\']=current[k];frame[k+\'_agreement\']=agreement[:,di]\n        member_path=output/\'LMS_STABILITAS/ANGGOTA\'/(\'rep_%02d_\' % rep+re.sub(r\'[^A-Za-z0-9]+\',\'_\',method.replace(\'++\',\'_PP\'))+\'.csv.gz\')\n        save_csv(member_path,frame);paths.append(member_path)\n    cache_write(files,path,signature,paths,{\'summary\':clean(row),\'identity\':identity})\n    return row,\'CALCULATE\'\n\n\ndef summarize_stability(rows):\n    raw=pd.DataFrame(rows);summary=[]\n    metrics=[\'mean_direction_agreement\',\'all_four_direction_agreement\',\'base_ls_agreement\',\'unresolved_learner_rate\',\n             \'joint_resolved_dimensional_coverage\',\'n_resolved_profiles\',\'ari\',\'nmi\',\'ami\']+[k+\'_agreement\' for k in KEYS]\n    for method in ORDER:\n        frame=raw.loc[raw.method.eq(method)]\n        if len(frame)!=30 or frame.rep.nunique()!=30: raise RuntimeError(\'Stabilitas harus berisi 30 resample per metode.\')\n        row={\'method\':method,\'n_resamples\':30}\n        for metric in metrics:\n            row[metric+\'_mean\']=frame[metric].mean();row[metric+\'_sd\']=frame[metric].std(ddof=1)\n            row[metric+\'_min\']=frame[metric].min();row[metric+\'_max\']=frame[metric].max()\n        summary.append(row)\n    paired=[];care=raw.loc[raw.method.eq(\'CARE-KM\')].set_index(\'rep\')\n    for method in [m for m in ORDER if m!=\'CARE-KM\']:\n        other=raw.loc[raw.method.eq(method)].set_index(\'rep\').loc[care.index]\n        for metric in [\'mean_direction_agreement\',\'base_ls_agreement\',\'ari\']:\n            delta=care[metric]-other[metric]\n            paired.append({\'baseline\':method,\'metric\':metric,\'care_minus_baseline_mean\':delta.mean(),\'sd_matched_differences\':delta.std(ddof=1),\n                           \'wins\':int((delta>1e-12).sum()),\'ties\':int((delta.abs()<=1e-12).sum()),\'losses\':int((delta<-1e-12).sum()),\n                           \'interpretation\':\'descriptive matched overlapping resamples; not 30 independent cohorts\'})\n    return raw,pd.DataFrame(summary),pd.DataFrame(paired)\n\n\ndef figures(output,ils,stability):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    paths=[];folder=output/\'GAMBAR\';folder.mkdir(parents=True,exist_ok=True)\n    data=ils.set_index(\'method\').loc[ORDER];y=np.arange(len(data))\n    fig,ax=plt.subplots(figsize=(8.5,5))\n    values=data.balanced_accuracy_mean.to_numpy()*100\n    # Percentile CIs need not contain the point estimate; draw their actual endpoints.\n    low=data.balanced_accuracy_ci95_low.to_numpy()*100;high=data.balanced_accuracy_ci95_high.to_numpy()*100\n    ax.hlines(y,low,high,color=\'#333333\');ax.plot(values,y,\'o\',color=\'#185785\')\n    ax.set_yticks(y,ORDER);ax.invert_yaxis();ax.set_xlabel(\'Mean balanced accuracy (%)\');ax.set_xlim(0,100)\n    ax.set_title(\'Pendikom holdout: direct profiles and paired learner bootstrap CIs\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'01_ils_mean_balanced_accuracy.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5));groups=[stability.loc[stability.method.eq(m),\'mean_direction_agreement\'].to_numpy()*100 for m in ORDER]\n    ax.boxplot(groups,positions=np.arange(1,9),widths=.55)\n    ax.set_xticks(np.arange(1,9),ORDER,rotation=30,ha=\'right\');ax.set_ylim(0,100)\n    ax.set_ylabel(\'Mean agreement of four directions (%)\');ax.set_title(\'LMS semantic stability across 30 matched 80% resamples\');fig.tight_layout()\n    for ext in [\'png\',\'pdf\']:\n        path=folder/(\'02_lms_semantic_stability.\'+ext);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n    plt.close(fig)\n    return paths\n\n\ndef unchanged(files,watched,code_checks):\n    for path,checksum in watched.items():\n        if files.hash(path)!=checksum: raise RuntimeError(\'Masukan berubah selama eksekusi: \'+path)\n    for stem,tag,var,expected in code_checks:\n        if notebook_hash(files,stem,tag,var)!=expected: raise RuntimeError(\'Kode notebook berubah selama eksekusi: \'+stem)\n\n\ndef run_semantics(root,configuration,embedded_source):\n    root=Path(root).expanduser().resolve();files=Files(root);cfg=check_config(configuration)\n    protocol,rules,watched=input_setup(files)\n    own_hash=notebook_hash(files,STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\')\n    code_checks=[(UPSTREAM,\'initializer25_configuration\',\'KONFIGURASI_25\',protocol[\'code\'][\'notebook_code\']),\n                 (STAGE,\'semantic26_configuration\',\'KONFIGURASI_26\',own_hash)]\n    code_state={\'implementation_sha256\':sha(embedded_source.encode(\'utf-8\')),\'notebook_code\':own_hash,\'runtime\':runtime()}\n    output=root/\'OUTPUT/TAMBAHAN\'/STAGE;control=output/\'KONTROL\'\n    output.mkdir(parents=True,exist_ok=True);control.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code_state,\'upstream_protocol\':protocol,\n        \'semantic_rule\':\'mode-first; member tie-break; unchanged member-derived 11-unit magnitude\',\n        \'primary_metric\':\'Pendikom mean balanced accuracy\',\'stability_reference\':\'own initializer full run 0; no reference chosen by outcomes\',\n        \'method_count\':8,\'mapping_jobs_expected\':432,\'stability_jobs_expected\':240,\'bootstrap_unit\':\'learner; all methods/runs/dimensions paired\',\n        \'unknown_vs_balanced\':\'UNRESOLVED direction differs from Balanced preference magnitude {1,3}\',\n        \'no_new_clustering\':True})\n    for ds in [\'LMS\',\'PENDIKOM\']: save_csv(output/\'PROTOKOL\'/(\'02_evidence_\'+ds+\'.csv\'),rules[ds])\n    counts={\'CACHE\':0,\'MAP\':0};stability_counts={\'CACHE\':0,\'CALCULATE\':0};rows=[];artifacts=[];stable=[];references={}\n    predictions={m:[] for m in METHODS};reference=None;truth=None;all_watched=dict(watched)\n    try:\n        for ci,context in enumerate(contexts(files),1):\n            unchanged(files,watched,code_checks)\n            print(f"[{ci}/33] {context[\'scope\']} | rep={context[\'rep\']} | n={len(context[\'X\'])}",flush=True)\n            for method in METHODS:\n                runs=15 if method in STOCHASTIC and context[\'scope\']!=\'LMS_RESAMPLING\' else 1\n                for run in range(runs):\n                    upstream=upstream_job(files,protocol,context,method,run);all_watched.update(upstream[\'watched\'])\n                    mapping,evidence,record,summary,disposition=semantic_job(files,output,control,context,upstream,rules[context[\'dataset\']],code_state,cfg)\n                    counts[disposition]+=1;rows.append(summary)\n                    artifacts.append({**upstream[\'identity\'],\'upstream_record_sha256\':upstream[\'checksum\'],\n                                      \'files\':json.dumps(list(record[\'outputs\']),ensure_ascii=False),\n                                      \'upstream_files\':json.dumps(list(upstream[\'record\'][\'outputs\']),ensure_ascii=False)})\n                    if context[\'scope\']==\'LMS_FULL\' and run==0:\n                        frame=profiles(mapping,upstream[\'labels\'],context[\'ids\'],context[\'indices\'])\n                        references[method]={\'profiles\':frame,\'signature\':digest({\'upstream\':upstream[\'checksum\'],\'mapping\':mapping.to_dict(\'records\')})}\n                    if context[\'scope\']==\'LMS_RESAMPLING\':\n                        row,dispo=stability_job(files,output,control,context,upstream,mapping,references[method],code_state,cfg)\n                        stable.append(row);stability_counts[dispo]+=1\n                    if context[\'scope\']==\'PENDIKOM_TRAIN\':\n                        if reference is None: reference,truth=ils_reference(files,context)\n                        predictions[method].append(numeric_profiles(profiles(mapping,upstream[\'test_labels\'],context[\'test_ids\'])))\n            save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'completed_mapping_jobs\':len(rows),\'mapping_counts\':counts,\'last_context\':context[\'scope\']})\n        if len(rows)!=432 or len(stable)!=240: raise RuntimeError(\'Jumlah partisi/stabilitas belum lengkap.\')\n        unchanged(files,all_watched,code_checks)\n        ils,ils_cached=evaluate_ils(files,output,control,predictions,reference,truth,code_state,cfg)\n        raw,stability,paired=summarize_stability(stable)\n        save_csv(output/\'01_ringkasan_semua_partisi.csv\',pd.DataFrame(rows))\n        save_csv(output/\'02_peta_artefak.csv\',pd.DataFrame(artifacts))\n        save_csv(output/\'LMS_STABILITAS/01_metrik_per_resample.csv\',raw)\n        save_csv(output/\'LMS_STABILITAS/02_ringkasan_8_initializer.csv\',stability)\n        save_csv(output/\'LMS_STABILITAS/03_perbedaan_berpasangan_deskriptif.csv\',paired)\n        save_json(output/\'LMS_STABILITAS/04_protokol_stabilitas.json\',{\n            \'reference\':\'own-method LMS full run 0 (seed 4161600), declared in stage 25\',\n            \'resampling\':\'30 fixed matched 80% samples; overlapping learners\',\n            \'inference\':\'descriptive mean/SD/range and matched differences; no independent-cohort p-values\',\n            \'unresolved\':\'never counted as correct direction agreement; conditional agreement includes paired coverage\',\n            \'assignment_source\':\'stage 25 bank; no reclustering\', \'compressed_members\':\'CSV inside gzip; learner IDs, profiles and agreements are auditable\'})\n        summary=pd.DataFrame(rows)\n        save_csv(output/\'LMS_FULL/01_keragaman_profil_per_run.csv\',summary.loc[summary.scope.eq(\'LMS_FULL\')])\n        paper=ils.set_index(\'method\').loc[ORDER].reset_index()\n        for metric in METRIC_NAMES:\n            paper[metric+\'_mean_ci95\']=paper.apply(lambda r: f"{100*r[metric+\'_mean\']:.2f} [{100*r[metric+\'_ci95_low\']:.2f}, {100*r[metric+\'_ci95_high\']:.2f}]",axis=1)\n        save_csv(output/\'TABEL/01_initializer_ils.csv\',paper[[\'method\',\'n_runs\']+[m+\'_mean_ci95\' for m in METRIC_NAMES]+[\'coverage_mean\',\'complete_profile_coverage\']])\n        save_csv(output/\'TABEL/02_initializer_stabilitas.csv\',stability[[\'method\',\'n_resamples\',\'mean_direction_agreement_mean\',\'mean_direction_agreement_sd\',\n            \'base_ls_agreement_mean\',\'base_ls_agreement_sd\',\'unresolved_learner_rate_mean\',\'n_resolved_profiles_mean\',\'ari_mean\']])\n        # Render only when figure inputs/code changed or a figure is missing/tampered.\n        fig_signature=digest({\'code\':code_state,\'ils\':sha((output/\'ILS/04_ringkasan_overall.csv\').read_bytes()),\n                              \'stability\':sha((output/\'LMS_STABILITAS/01_metrik_per_resample.csv\').read_bytes())})\n        fig_cache=cache_read(files,control/\'gambar.json\',fig_signature,cfg[\'force\'])\n        if not fig_cache: cache_write(files,control/\'gambar.json\',fig_signature,figures(output,ils,raw),{})\n        unchanged(files,all_watched,code_checks)\n        save_json(output/\'PROTOKOL/03_manifest_masukan.json\',all_watched)\n        status={\'status\':\'selesai\',\'completed_mapping_jobs\':432,\'completed_stability_jobs\':240,\'method_count\':8,\n                \'mapping_counts\':counts,\'stability_counts\':stability_counts,\'ils_metrics_cached\':ils_cached,\n                \'full_protocol\':cfg[\'bootstrap_repetitions\']>=10000,\'n_ils_test\':139,\n                \'new_clustering_runs\':0,\'output\':str(output),\'finished_utc\':datetime.now(timezone.utc).isoformat()}\n        # Manifest includes stage-26 scientific deliverables, not mutable control logs.\n        outputs={str(p.relative_to(root)).replace(\'\\\\\',\'/\'):sha(p.read_bytes()) for p in output.rglob(\'*\') if p.is_file() and control not in p.parents}\n        save_json(control/\'manifest_keluaran.json\',outputs);save_json(control/\'status_eksekusi.json\',status)\n        print(\'Tahap 26 selesai. Partisi:\',counts,\'| Stabilitas:\',stability_counts,\'| ILS cache:\',ils_cached,flush=True)\n        print(\'Hasil:\',output)\n        return {\'ils\':ils,\'stability\':stability,\'status\':status,\'output\':str(output)}\n    except BaseException as error:\n        save_json(control/\'status_eksekusi.json\',{\'status\':\'terhenti\',\'completed_mapping_jobs\':len(rows),\n                  \'mapping_counts\':counts,\'error\':type(error).__name__+\': \'+str(error),\n                  \'resume\':\'jalankan notebook 26 kembali; cache diperiksa berdasarkan isi\'})\n        raise\n'

_semantic26 = types.ModuleType('_semantic26')
exec(compile(SEMANTIC26_SOURCE, '_semantic26', 'exec'), _semantic26.__dict__)


In [4]:
HASIL_26 = _semantic26.run_semantics(ROOT_26, KONFIGURASI_26, SEMANTIC26_SOURCE)
display(HASIL_26['ils'])
display(HASIL_26['stability'][['method', 'mean_direction_agreement_mean',
                              'mean_direction_agreement_sd', 'base_ls_agreement_mean',
                              'unresolved_learner_rate_mean']])
print(HASIL_26['status'])
print('Folder hasil:', HASIL_26['output'])


[1/33] LMS_FULL | rep=None | n=6658
[2/33] LMS_RESAMPLING | rep=0 | n=5326
[3/33] LMS_RESAMPLING | rep=1 | n=5326
[4/33] LMS_RESAMPLING | rep=2 | n=5326
[5/33] LMS_RESAMPLING | rep=3 | n=5326
[6/33] LMS_RESAMPLING | rep=4 | n=5326
[7/33] LMS_RESAMPLING | rep=5 | n=5326
[8/33] LMS_RESAMPLING | rep=6 | n=5326
[9/33] LMS_RESAMPLING | rep=7 | n=5326
[10/33] LMS_RESAMPLING | rep=8 | n=5326
[11/33] LMS_RESAMPLING | rep=9 | n=5326
[12/33] LMS_RESAMPLING | rep=10 | n=5326
[13/33] LMS_RESAMPLING | rep=11 | n=5326
[14/33] LMS_RESAMPLING | rep=12 | n=5326
[15/33] LMS_RESAMPLING | rep=13 | n=5326
[16/33] LMS_RESAMPLING | rep=14 | n=5326
[17/33] LMS_RESAMPLING | rep=15 | n=5326
[18/33] LMS_RESAMPLING | rep=16 | n=5326
[19/33] LMS_RESAMPLING | rep=17 | n=5326
[20/33] LMS_RESAMPLING | rep=18 | n=5326
[21/33] LMS_RESAMPLING | rep=19 | n=5326
[22/33] LMS_RESAMPLING | rep=20 | n=5326
[23/33] LMS_RESAMPLING | rep=21 | n=5326
[24/33] LMS_RESAMPLING | rep=22 | n=5326
[25/33] LMS_RESAMPLING | rep=23 | n=532

,method,dimension,n_runs,n_test,bootstrap_valid,bootstrap_requested,coverage_mean,complete_profile_coverage,accuracy_mean,accuracy_sd_runs,accuracy_ci95_low,accuracy_ci95_high,balanced_accuracy_mean,balanced_accuracy_sd_runs,balanced_accuracy_ci95_low,balanced_accuracy_ci95_high,macro_f1_mean,macro_f1_sd_runs,macro_f1_ci95_low,macro_f1_ci95_high
0,Random,mean_four_dimensions,15,139,10000,10000,0.993885,0.975540,0.802638,0.036472,0.780695,0.824101,0.721997,0.039195,0.686027,0.757130,0.718101,0.042543,0.673600,0.754848
1,Huang,mean_four_dimensions,15,139,10000,10000,0.996283,0.985132,0.793165,0.024425,0.770624,0.815111,0.701004,0.029278,0.668817,0.732012,0.693733,0.037411,0.656375,0.724123
2,Cao,mean_four_dimensions,1,139,10000,10000,0.980216,0.920863,0.857914,NaN,0.827338,0.886691,0.769490,NaN,0.724434,0.814669,0.786755,NaN,0.731626,0.833500
3,Bai,mean_four_dimensions,1,139,10000,10000,1.000000,1.000000,0.848921,NaN,0.816547,0.879496,0.744197,NaN,0.692726,0.793687,0.758116,NaN,0.697496,0.807847
4,K-Modes++,mean_four_dimensions,15,139,10000,10000,0.992926,0.971703,0.813909,0.033468,0.793405,0.833573,0.741481,0.016292,0.707016,0.775165,0.737928,0.025011,0.695702,0.772366
5,HR-2023,mean_four_dimensions,15,139,10000,10000,0.994005,0.976019,0.787410,0.023472,0.765108,0.808633,0.698829,0.026255,0.666477,0.730226,0.691454,0.033959,0.654023,0.721824
6,CARE-KM,mean_four_dimensions,1,139,10000,10000,1.000000,1.000000,0.897482,NaN,0.870504,0.922662,0.812925,NaN,0.765235,0.857619,0.829776,NaN,0.776050,0.872808
7,BK-Modes,mean_four_dimensions,1,139,10000,10000,1.000000,1.000000,0.766187,NaN,0.733813,0.798561,0.719312,NaN,0.671208,0.766515,0.700942,NaN,0.647773,0.746870


,method,mean_direction_agreement_mean,mean_direction_agreement_sd,base_ls_agreement_mean,unresolved_learner_rate_mean
0,CARE-KM,0.779930,0.018507,0.424365,0.001208
1,Cao,0.775088,0.030636,0.422763,0.000000
2,Bai,0.783106,0.034715,0.429509,0.001227
3,BK-Modes,0.766556,0.031439,0.408074,0.000000
4,K-Modes++,0.730267,0.024875,0.309044,0.000000
5,Random,0.731728,0.019793,0.285799,0.003818
6,Huang,0.730016,0.020573,0.284016,0.006615
7,HR-2023,0.730059,0.021909,0.283715,0.004631


{'status': 'selesai', 'completed_mapping_jobs': 432, 'completed_stability_jobs': 240, 'method_count': 8, 'mapping_counts': {'CACHE': 0, 'MAP': 432}, 'stability_counts': {'CACHE': 0, 'CALCULATE': 240}, 'ils_metrics_cached': False, 'full_protocol': True, 'n_ils_test': 139, 'new_clustering_runs': 0, 'output': '/content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/26_Pengaruh_Initializer_Profil_Semantik', 'finished_utc': '2026-11-06T09:32:02.080473+00:00'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/26_Pengaruh_Initializer_Profil_Semantik


## Membaca hasil dan melanjutkan eksperimen

- `ILS/04_ringkasan_overall.csv`: mean accuracy, BA, macro-F1, variasi seed, dan CI 95% untuk delapan initializer.
- `ILS/03_ringkasan_per_dimensi.csv`: rincian empat dimensi.
- `ILS/01_prediksi_per_peserta.csv`, `05_confusion_matrices.csv`, dan `08_reference_counts.csv`: denominator serta prediksi yang dapat diaudit.
- `ILS/07_perbedaan_berpasangan_vs_care.csv`: selisih berpasangan dan intervalnya.
- `ILS/06_bootstrap_paired_metrics.npz`: hasil bootstrap berpasangan untuk analisis gabungan.
- `LMS_STABILITAS/`: 240 hasil per resample, ringkasan delapan initializer, dan perbandingan peserta dalam CSV terkompresi `.csv.gz`.
- `BANK/`: profil cluster dan evidence setiap partisi; untuk scope train tersedia target train/test yang bisa dipakai pada pemodelan berikutnya.
- `01_ringkasan_semua_partisi.csv` dan `02_peta_artefak.csv`: daftar 432 partisi dan hubungan dengan bank sumber tahap 25.
- `TABEL/`, `GAMBAR/`, `PROTOKOL/`: tabel, gambar PNG/PDF, konfigurasi dan manifest.
- `KONTROL/`: checkpoint dan status eksekusi.

Profil peserta full/resample dapat direkonstruksi dengan menggabungkan
mapping cluster di bank 26 dan assignment sumber di bank 25; sumber
assignment dicatat pada peta artefak agar data tidak disalin berulang.
